# J/psi to mu mu, 2023: basic muons and electrons

The first study: run the processor over a file of a few samples, print the cutflow of
the `baseline` channel, and plot the basic histograms of the muons and electrons with the samples
overlaid. The object cuts are the ones `baseline` applies in `configs/selections.yaml`;
the cell that builds the processor prints them.

Rerun it after changing an object, a cut or a histogram: it is the quickest picture of
what the change did, and its layout is one to copy for a new study: configure, run,
then one `plot_row` per group of histograms.

In [ ]:
# python
import os
import sys
import importlib
import importlib.util
# columnar analysis and plotting
import numpy as np
import matplotlib.pyplot as plt
from coffea import processor

# the analysis package: the repository this notebook is in (found by walking up from
# it), or else where the package is installed (pip install -e .)
REPO = os.path.abspath(os.getcwd())
while not os.path.isfile(os.path.join(REPO, "jpsi_analysis", "__init__.py")):
    if os.path.dirname(REPO) == REPO:
        spec = importlib.util.find_spec("jpsi_analysis")
        if spec is None or not spec.origin:
            raise RuntimeError("open this notebook from inside the analysis repository, "
                               "or install the package (pip install -e .)")
        REPO = os.path.dirname(os.path.dirname(spec.origin))
        break
    REPO = os.path.dirname(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# local
from jpsi_analysis import TREE_NAME
from jpsi_analysis.tools import utilities, plotting, selection, histogram, cutflow, schema
from jpsi_analysis.tools import processor as analysis_processor
from jpsi_analysis.definitions import objects, cuts, hists, weights
# reload the local modules, to pick up edits made while the notebook is open
for module in (utilities, cutflow, histogram, schema, objects, cuts, hists, weights,
               selection, analysis_processor, plotting):
    importlib.reload(module)
plotting.set_plot_style()
%matplotlib inline

## Samples

Every sample under `configs/samples/`, config by config and group by group.

In [ ]:
for config, tag, group in utilities.find_samples():
    names = list(group["samples"])
    print(f"{config}, {tag} ({group.get('path') or 'complete paths'}): {len(names)} sample(s)")
    for name in names[:10]:
        print("   ", name)
    if len(names) > 10:
        print(f"    ... and {len(names) - 10} more")

In [ ]:
samples = ["jpsi_mumu_2023_condor"]   # simulation and data to compare: any names from the list above
fileset = utilities.make_fileset(samples, max_files=1)   # one file each; max_files=-1 reads all

## Process

The baseline selection, and the histogram collections of the objects plotted below.

In [ ]:
channels = ["baseline"]
hist_collections = ["muon_base", "electron_base"]

p = analysis_processor.AnalysisProcessor(channels, hist_collections)
selection_cuts = p.build_cuts()["baseline"]
for obj, cut_names in selection_cuts["obj"].items():
    print(f"{obj}: {cut_names}")
print("event cuts:", selection_cuts["evt"])

runner = processor.Runner(
    executor=processor.IterativeExecutor(),
    # executor=processor.FuturesExecutor(workers=4),   # several local processes
    schema=schema.AnalysisSchema,
    chunksize=100_000,
    # maxchunks=1,          # the first chunk of each sample only, for a quicker look
    skipbadfiles=True,    # carry on past a file that cannot be read: see what was read below
    metadata_cache={},    # read every sample's is_data, year and skim_factor afresh
)
out = runner(fileset, processor_instance=p, treename=TREE_NAME)
for sample in samples:
    print(sample, f"{out[sample]['metadata']['n_evts']} events" if sample in out else "nothing read")

## Cutflow

Events left after each event cut of `baseline`, the samples side by side. These are raw
counts: weighted ones need cross sections and a luminosity in `configs/`.

In [ ]:
read = [sample for sample in samples if sample in out]
print("\n".join(utilities.cutflow_table(out, "baseline", samples=read, weighted=False)))

## Basic histograms

`plot_row` draws one panel per histogram with every sample overlaid, each normalised to
unit area so that simulation and data compare in shape. Where the framework has the
invariant mass of a pair, it is drawn too, as events per bin on a logarithmic axis from
0.2 GeV to 1 TeV, on which any resonance of the pair shows as a peak. The figures are also saved in
`output/studies/` at the top of the repository (git ignores `output/`).

In [ ]:
def plot_row(names, channel="baseline", density=True, ylog=False):
    """One panel per histogram, the samples overlaid."""
    fig, axes = plt.subplots(1, len(names), figsize=(12 * len(names), 10))
    for ax, name in zip(np.atleast_1d(axes), names):
        plt.sca(ax)
        plotting.plot_samples(out, name, channel, samples=read, density=density)
        if ylog:
            ax.set_yscale("log")
    return fig


figures = os.path.join(REPO, "output", "studies")
os.makedirs(figures, exist_ok=True)

In [ ]:
fig = plot_row(["muon_pt", "electron_pt"])
fig.savefig(os.path.join(figures, "basic_pt.png"))

In [ ]:
fig = plot_row(["muon_n", "electron_n"])
fig.savefig(os.path.join(figures, "basic_multiplicity.png"))

In [ ]:
# where the muons are, in the first sample
plt.figure(figsize=(12, 10))
plotting.plot(utilities.get_hist(out, read[0], "muon_eta_phi", "baseline"))
plt.savefig(os.path.join(figures, "muon_eta_phi.png"))

In [ ]:
# the invariant mass of the two leading muons, on a logarithmic axis: a resonance
# of the pair shows as a peak wherever it is (J/psi, Upsilon, Z, ...). Events per bin, not a
# density, which these narrow bins would raise at low mass; and no error bars, since most
# bins are empty and the error bar of an empty bin is not zero
fig, ax = plt.subplots(figsize=(12, 10))
plotting.plot_samples(out, "muon_muon_invmass", "baseline", samples=read, yerr=False)
ax.set_xscale("log")
ax.set_ylabel("Events per bin")
fig.savefig(os.path.join(figures, "muon_muon_invmass.png"))

## Next

- A tighter or different object cut: define it in `jpsi_analysis/definitions/cuts.py`, list it
  under the object in `jpsi_analysis/configs/selections.yaml`, rerun.
- Another histogram: define it in `jpsi_analysis/definitions/hists.py`, add it to a
  collection in `jpsi_analysis/configs/hist_collections.yaml`, add the collection above.
- All files of a sample: `max_files=-1` (with `processor.FuturesExecutor`, or the
  `scaleout` component for a cluster).